In [1]:
# imports
import pandas as pd
import requests
import yfinance as yf
import numpy as np
from transformers import pipeline
from datetime import datetime, timedelta
from dotenv import load_dotenv
import os
from edgar import Company, set_identity
import regex as re
import torch
import time

DATA_DIR = '../../../data'            # shared raw inputs and caches
OUT_DIR = '../../../outputs/v3/data'  # everything this notebook writes
os.makedirs(OUT_DIR, exist_ok=True)

### Gathering Data

In [2]:
# Universe: S&P 500. The list is saved on the first run and reused afterwards,
# so reruns use the same tickers the EODHD news was downloaded for.
UNIVERSE_PATH = f'{OUT_DIR}/sp500_tickers.csv'
if os.path.exists(UNIVERSE_PATH):
    table = pd.read_csv(UNIVERSE_PATH)
else:
    headers = {"User-Agent": "Mozilla/5.0 (compatible; my-script/1.0)"}
    url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
    html = requests.get(url, headers=headers).text
    table = pd.read_html(html)[0]
    table['Symbol'] = table['Symbol'].str.replace('.', '-').tolist()
    table = table[['Symbol', 'Security', 'GICS Sector', 'GICS Sub-Industry']]
    table.to_csv(UNIVERSE_PATH, index=False)
table.head()

,Symbol,Security,GICS Sector,GICS Sub-Industry
0,MMM,3M,Industrials,Industrial Conglomerates
1,AOS,A. O. Smith,Industrials,Building Products
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment
3,ABBV,AbbVie,Health Care,Biotechnology
4,ACN,Accenture,Information Technology,IT Consulting & Other Services


#### Tickers
The universe is the S&P 500 (current constituents, saved to `sp500_tickers.csv`).

The original 19 AI-sector tickers are kept in `ai_tickers` for subsample comparisons:<br>
**Semiconductors**: NVDA (NVIDIA), AMD (Advanced Micro Devices), INTC (Intel), ASML (ASML Holding), TSM (Taiwan Semiconductor Manufacturing), AVGO (Broadcom),
    MRVL (Marvell Technology), ARM (Arm Holdings), SMCI (Super Micro Computer)<br>
**Hyperscalers**: MSFT (Microsoft), GOOGL (Google), AMZN (Amazon), META (Facebook) <br>
**AI Pure Plays**: PLTR (Palantir), AI (C3.ai), SOUN (Soundhound AI) <br>
**Infrastructure**: DELL (Dell Technologies), ANET (Arista Networks), HPE (Hewlett Packard Enterprise)

In [3]:
# list of all the tickers with a dictionary that sorts them by type of stock
tickers = pd.read_csv(f'{DATA_DIR}/tickers.csv')
tickers = tickers.drop(columns=['Unnamed: 0'])

ai_tickers = {
    'semis':   ['NVDA','AMD','INTC','ASML','TSM','AVGO','MRVL','ARM','SMCI'], # semiconductors
    'hypers':  ['MSFT','GOOGL','AMZN','META'], # hyperscalar
    'pure':    ['PLTR','AI','SOUN'], # AI pure plays
    'infra':   ['DELL','ANET','HPE'] # infrastructure
}
ai_tickers_flat = [t for group in ai_tickers.values() for t in group]
all_tickers = table['Symbol'].tolist()
print(f'{len(all_tickers)} tickers')

503 tickers


In [ ]:
# Data from 2018 to today: 2019-2025 is the development sample, 2026 the holdout
prices = yf.download(
    tickers=all_tickers + ['SOXX', 'QQQ'],
    start='2018-01-01',
    end=(datetime.today() + timedelta(days=1)).strftime('%Y-%m-%d'),
    auto_adjust=True,
    progress=False
)
prices.to_csv(f'{OUT_DIR}/prices_raw.csv')

In [5]:
prices.head()

Price      Adj Close                Close                             \
Ticker          FDXF HONA VYLR          A       AAPL       ABBV ABNB   
Date                                                                   
2018-01-02       NaN  NaN  NaN  63.306446  40.201828  67.870384  NaN   
2018-01-03       NaN  NaN  NaN  64.917191  40.194832  68.932442  NaN   
2018-01-04       NaN  NaN  NaN  64.430206  40.381535  68.539322  NaN   
2018-01-05       NaN  NaN  NaN  65.460358  40.841290  69.732460  NaN   
2018-01-08       NaN  NaN  NaN  65.600822  40.689587  68.615204  NaN   

Price                                         ...   Volume                    \
Ticker            ABT       ACGL         ACN  ...       WY     WYNN      XEL   
Date                                          ...                              
2018-01-02  49.841595  27.988113  132.881607  ...  3144000  2581200  2443400   
2018-01-03  49.951801  28.013470  133.494965  ...  2922800  3530900  4114900   
2018-01-04  49.867008  28.118069  135.075623  ...  2727800  2708800  2807000   
2018-01-05  50.011135  28.007132  136.189880  ...  2924800  2186600  3728000   
2018-01-08  49.867008  28.010302  137.278183  ...  2760800  1803800  3837600   

Price                                                                      
Ticker           XOM     XYL       XYZ      YUM      ZBH    ZBRA      ZTS  
Date                                                                       
2018-01-02  11469300  877800  11240500  1747800  1818259  310600  2135600  
2018-01-03  13957700  778800  11617900  2554900  1368664  253000  2328200  
2018-01-04  10863000  796500  20469800  1971200  1105396  435200  2534000  
2018-01-05  11047600  666800  31102000  1927100  1095302  301800  2166100  
2018-01-08  10927100  554700  26626100  1599400  1286985  218600  3631400  

[5 rows x 2528 columns]

### Cleaning Data

In [6]:
close  = prices['Close']
volume = prices['Volume']
soxx   = close['SOXX']
qqq    = close['QQQ']

In [7]:
# Remove tickers with more than 5% NaN values
clean_prices = close.loc[:, close.isna().mean() <= 0.05]

# Forward fill short gaps of 1-2 days
clean_prices = clean_prices.ffill(limit=2)
clean_prices = clean_prices.dropna()
clean_prices.to_csv(f'{OUT_DIR}/clean_prices.csv')

In [8]:
# Convert prices to log
log_ret = np.log(clean_prices / clean_prices.shift(1)).dropna()

### Feature Engineering

In [9]:
# SOXX: holds 30 largest semiconductor companies
# QQQ: holds 100 largest non-financial companies on the NASDAQ, heavily weighted towards big tech
def make_features(close: pd.Series,
                     volume: pd.Series,
                     soxx: pd.Series,
                     qqq: pd.Series) -> pd.DataFrame:
    df = pd.DataFrame(index=close.index)

    # Lag returns: percent change over x days
    for lag in [1, 3, 5, 10, 20]:
        df[f'ret_{lag}d'] = close.pct_change(lag)

    # Rolling volatility: std of returns
    df['vol_5d']  = close.pct_change().rolling(5).std()
    df['vol_20d'] = close.pct_change().rolling(20).std()

    # RSI (14-day): Relative Strength Index; measures magnitude and recent price changes in stocks
    # whether a stock has been moving up or down more forcefully over its recent trading days
    # > 70: overbought (a lot of upward activity)
    # < 30: oversold (a lot of downward activity)
    # 50: roughly equal up and down movement
    delta = close.pct_change()
    gain  = delta.clip(lower=0).rolling(14).mean() # average of up days
    loss  = (-delta.clip(upper=0)).rolling(14).mean() # average of down days
    df['rsi_14'] = 100 - (100 / (1 + gain / loss))

    # How does the stock compare to SOXX and QQQ?
    df['rel_soxx'] = close.pct_change(20) - soxx.pct_change(20)
    df['rel_qqq']  = close.pct_change(20) - qqq.pct_change(20)

    # Volume spike: number of shares traded on a given day
    df['vol_spike'] = volume / volume.rolling(20).mean()

    # 52-week high proximity: the highest price traded at over the last 252 trading days (1 year)
    df['pct_from_52wk_high'] = close / close.rolling(252).max() - 1

    return df.dropna()

In [10]:
all_feature_dfs = []
for ticker in all_tickers:
    if ticker not in close.columns:
        continue
    ticker_feats = make_features(
        close  = close[ticker],
        volume = volume[ticker],
        soxx   = soxx,
        qqq    = qqq
    )
    ticker_feats['Ticker'] = ticker      
    all_feature_dfs.append(ticker_feats)

/var/folders/5l/vsm1wvjn4qdg25qwjfcj83kr0000gn/T/ipykernel_73869/861326954.py:11: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  df[f'ret_{lag}d'] = close.pct_change(lag)
/var/folders/5l/vsm1wvjn4qdg25qwjfcj83kr0000gn/T/ipykernel_73869/861326954.py:11: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  df[f'ret_{lag}d'] = close.pct_change(lag)
/var/folders/5l/vsm1wvjn4qdg25qwjfcj83kr0000gn/T/ipykernel_73869/861326954.py:11: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling p

In [11]:
features_df = pd.concat(all_feature_dfs).reset_index()
features_df = features_df.set_index(['Date','Ticker'])

In [12]:
# Forward 5-day cumulative return
features_df['fwd_5d'] = (
    features_df.groupby('Ticker')['ret_1d']
    .transform(lambda x: x.rolling(5).sum().shift(-5))
)

features_df = features_df.dropna(subset=['fwd_5d'])

# Cross-sectional: top half = 1, bottom half = 0 (guaranteed 50/50 per date)
# Which stocks will outperform their peers? 
features_df['Target'] = (
    features_df.groupby('Date')['fwd_5d']
    .transform(lambda x: (x > x.median()).astype(int))
)

In [13]:
features_df.head()

,,ret_1d,ret_3d,ret_5d,ret_10d,ret_20d,vol_5d,vol_20d,rsi_14,rel_soxx,rel_qqq,vol_spike,pct_from_52wk_high,fwd_5d,Target
Date,Ticker,,,,,,,,,,,,,,
2019-01-02,MMM,0.002152,0.001311,0.069029,-0.009698,-0.081618,0.019782,0.019320,43.774159,-0.022212,0.001318,0.920058,-0.242101,0.008613,0
2019-01-03,MMM,-0.037654,-0.029625,-0.013422,-0.055510,-0.119755,0.022619,0.020605,35.231491,0.017180,0.008379,1.228533,-0.270639,0.053027,0
2019-01-04,MMM,0.041141,0.004094,0.003251,0.007159,-0.053808,0.028271,0.022245,41.589701,-0.001133,0.000760,1.089791,-0.240632,0.004706,0
2019-01-07,MMM,-0.002300,-0.000366,0.007974,0.018679,-0.058219,0.028019,0.022215,45.779948,-0.027993,-0.008200,0.798790,-0.242379,0.006798,0
2019-01-08,MMM,0.004191,0.043100,0.005983,0.043156,-0.033091,0.027958,0.021794,49.588140,-0.036040,-0.024394,0.912325,-0.239204,-0.014201,0


In [14]:
features_df.to_csv(f'{OUT_DIR}/features_df.csv')

### News Sentiment (EODHD)
News is scored outside this notebook, because scoring ~500 tickers takes hours. Run these from the repository root, **after** the cell above saves `features_df.csv`:

```
caffeinate -i python scripts/download_news.py > logs/download_news.log 2>&1      # already done
caffeinate -i python scripts/score_sentiment.py > logs/score_sentiment.log 2>&1
```

`score_sentiment.py` assigns each article to the first trading day whose 4pm ET close comes after it was published, and writes `outputs/v3/data/sentiment_df.csv`.

In [15]:
# merge sentiment df back to features df
sentiment_df = pd.read_csv(f'{OUT_DIR}/sentiment_df.csv', parse_dates=['Date'])
news_cols = ['news_sentiment', 'news_count', 'pct_positive', 'eod_sentiment']

df = features_df.reset_index()
df['Date'] = pd.to_datetime(df['Date'])
df = df.merge(sentiment_df, on=['Date', 'Ticker'], how='left')
df = df.set_index(['Date', 'Ticker'])

# Fill missing sentiment (days with no news)
df[news_cols] = df[news_cols].fillna(0)

### SEC 8K Filings (Earnings Releases)

Filings are downloaded and scored by `scripts/score_filings.py` (zero-shot topic classification with DeBERTa, plus FinBERT tone on AI sentences). Run it from the repository root after `sp500_tickers.csv` exists:

```
python3 scripts/score_filings.py --time-only NVDA MMM                       # estimate run time
caffeinate -i python3 scripts/score_filings.py > logs/score_filings.log 2>&1
```

It saves each filing's text and scores per ticker, so it can be stopped and restarted, and writes `outputs/v3/data/filing_signals_df.csv`.

In [ ]:
# Filing-level signals from scripts/score_filings.py
transcript_signals_df = pd.read_csv(f'{OUT_DIR}/filing_signals_df.csv')
text_cols = [col for col in transcript_signals_df.columns if col not in ('Ticker', 'Date', 'Accepted')]

# Earnings 8-Ks are often filed after the 4pm ET close. Count each filing on the first trading
# day whose close comes after the SEC accepted it (same rule as score_sentiment.py uses for news).
accepted = pd.to_datetime(transcript_signals_df['Accepted'], utc=True).dt.tz_convert('America/New_York')
day = accepted.dt.tz_localize(None).dt.normalize()
day = day + pd.to_timedelta((accepted.dt.hour >= 16).astype(int), unit='D')
trading_days = features_df.index.get_level_values('Date').unique().sort_values()
idx = np.searchsorted(trading_days.values, day.values.astype('datetime64[ns]'), side='left')
transcript_signals_df = transcript_signals_df[idx < len(trading_days)]    # filed after the panel ends
transcript_signals_df['Date'] = trading_days[idx[idx < len(trading_days)]]

transcript_signals_df = (transcript_signals_df.drop(columns='Accepted')
                         .set_index(['Date', 'Ticker']).sort_index())

transcript_signals_df.to_csv(f'{OUT_DIR}/transcript_signals_df.csv')
transcript_signals_df.head()

In [ ]:
# merge back to original dataframe
df = df.reset_index()

df = pd.merge_asof(
    df.sort_values('Date'),
    transcript_signals_df.reset_index().rename(columns={'date': 'Date'}).sort_values('Date'),
    on='Date',
    by='Ticker',
    direction='backward'
)

with_domain_keywords = df.set_index(['Date', 'Ticker'])
with_domain_keywords[text_cols] = with_domain_keywords[text_cols].fillna(0)

### Final Cleaning

In [ ]:
edgar_cols = text_cols

In [30]:
with_domain_keywords[edgar_cols] = with_domain_keywords[edgar_cols].fillna(0)

In [31]:
with_domain_keywords.to_csv(f'{OUT_DIR}/ai_sector_features_df.csv')

In [32]:
with_domain_keywords.head()

ret_1d    ret_3d    ret_5d   ret_10d   ret_20d    vol_5d  \
Date       Ticker                                                               
2019-01-02 MMM     0.002152  0.001311  0.069029 -0.009698 -0.081618  0.019782   
           DE     -0.007173  0.014661  0.083645  0.019107 -0.038797  0.021602   
           BSX    -0.024901 -0.000580  0.086038  0.012041 -0.085214  0.031285   
           SO     -0.004554 -0.005233  0.016271 -0.049565 -0.076273  0.007523   
           LEN     0.020690  0.011646  0.049921 -0.001500 -0.064826  0.018326   

                    vol_20d     rsi_14  rel_soxx   rel_qqq  ...  \
Date       Ticker                                           ...   
2019-01-02 MMM     0.019320  43.774159 -0.022212  0.001318  ...   
           DE      0.028384  53.267067  0.020608  0.044139  ...   
           BSX     0.028555  41.570152 -0.025808 -0.002277  ...   
           SO      0.014076  22.247292 -0.016867  0.006663  ...   
           LEN     0.024267  47.707249 -0.005420  0.018111  ...   

                   eod_sentiment  nvidia_mentions  amd_mentions  \
Date       Ticker                                                 
2019-01-02 MMM               0.0              0.0           0.0   
           DE                0.0              0.0           0.0   
           BSX               0.0              0.0           0.0   
           SO                0.0              0.0           0.0   
           LEN               0.0              0.0           0.0   

                   intel_mentions  gpu_mentions  capex_up_score  \
Date       Ticker                                                 
2019-01-02 MMM                0.0           0.0         0.00000   
           DE                 0.0           0.0         0.16488   
           BSX                0.0           0.0         0.00000   
           SO                 0.0           0.0         0.00000   
           LEN                0.0           0.0         0.00000   

                   capex_down_score  competitor_mentions  capex_net  \
Date       Ticker                                                     
2019-01-02 MMM                  0.0                  0.0    0.00000   
           DE                   0.0                  0.0    0.16488   
           BSX                  0.0                  0.0    0.00000   
           SO                   0.0                  0.0    0.00000   
           LEN                  0.0                  0.0    0.00000   

                   ai_sentence_ratio  
Date       Ticker                     
2019-01-02 MMM                   0.0  
           DE                    0.0  
           BSX                   0.0  
           SO                    0.0  
           LEN                   0.0  

[5 rows x 27 columns]

In [33]:
missing = set(all_tickers) - set(transcript_signals_df.index.
    get_level_values('Ticker').unique())
print(missing)
# {'ARM', 'TSM', 'ASML'}

{'FDXF', 'HONA', 'ECHO', 'VYLR'}
